# JAX: jit, vmap, grad

**Interview focus:** JAX transforms — `jit` for compilation, `vmap` for batching, `grad` for autodiff.

In [ ]:
import jax
import jax.numpy as jnp
from jax import grad, jit, vmap, value_and_grad
from jax import random

## 1. `grad` — Automatic Differentiation

In [ ]:
def f(x):
  return x ** 3 + 2 * x

df_dx = grad(f)
print(f"f(3) = {f(3.0)}")
print(f"f'(3) = {df_dx(3.0)}")  # 3*9 + 2 = 29

In [ ]:
# grad works on the first argument by default
def loss_fn(params, x, y):
  pred = params['w'] * x + params['b']
  return jnp.mean((pred - y) ** 2)

grad_fn = grad(loss_fn)  # gradients w.r.t. params (first arg)

params = {'w': jnp.array(1.0), 'b': jnp.array(0.0)}
x = jnp.array([1.0, 2.0, 3.0])
y = jnp.array([2.0, 4.0, 6.0])

grads = grad_fn(params, x, y)
print(f"grads: {grads}")

In [ ]:
# value_and_grad — compute loss and gradients in one pass
val_and_grad = value_and_grad(loss_fn)
loss, grads = val_and_grad(params, x, y)
print(f"loss: {loss:.4f}, grads: {grads}")

## 2. `jit` — Just-In-Time Compilation

In [ ]:
import time

def slow_matmul(A, B):
  return A @ B

fast_matmul = jit(slow_matmul)

A = random.normal(random.PRNGKey(0), shape=(2000, 2000))
B = random.normal(random.PRNGKey(1), shape=(2000, 2000))

# Warm up (compilation happens on first call)
_ = fast_matmul(A, B).block_until_ready()

start = time.time()
result = fast_matmul(A, B).block_until_ready()
print(f"jit matmul: {time.time() - start:.4f}s")

## 3. `vmap` — Vectorizing over Batch Dimensions

In [ ]:
# Without vmap: loop over batch
def predict_single(x, w, b):
  return jnp.dot(w, x) + b

# With vmap: vectorize over the batch dimension (axis 0)
predict_batch = vmap(predict_single, in_axes=(0, None, None))

X = random.normal(random.PRNGKey(0), shape=(32, 10))  # 32 samples, 10 features
w = random.normal(random.PRNGKey(1), shape=(10,))
b = jnp.array(0.5)

predictions = predict_batch(X, w, b)
print(f"predictions shape: {predictions.shape}")  # (32,)

In [ ]:
# vmap over multiple axes
def pairwise_dist(a, b):
  return jnp.sum((a - b) ** 2)

dist_matrix = vmap(vmap(pairwise_dist, in_axes=(None, 0)), in_axes=(0, None))

points = random.normal(random.PRNGKey(0), shape=(5, 3))
dists = dist_matrix(points, points)
print(f"distance matrix shape: {dists.shape}")
print(dists.round(2))

## 4. Composing Transforms

In [ ]:
# jit(grad(f)) — compiled gradient
def loss(params, x, y):
  pred = jnp.dot(x, params['w']) + params['b']
  return jnp.mean((pred - y) ** 2)

compiled_grad = jit(grad(loss))

params = {'w': jnp.zeros(5), 'b': jnp.array(0.0)}
x = random.normal(random.PRNGKey(0), shape=(100, 5))
y = random.normal(random.PRNGKey(1), shape=(100,))

grads = compiled_grad(params, x, y)
print(grads)

In [ ]:
# jit(vmap(grad(...))) — batched compiled gradients
def per_sample_loss(params, x_i, y_i):
  pred = jnp.dot(x_i, params['w']) + params['b']
  return (pred - y_i) ** 2

batched_grad = jit(vmap(grad(per_sample_loss), in_axes=(None, 0, 0)))
grads = batched_grad(params, x, y)
print(f"per-sample grad w shape: {grads['w'].shape}")  # (100, 5)

## 5. Higher-Order Derivatives

In [ ]:
def g(x):
  return x ** 4

g_prime = grad(g)       # 4x^3
g_double_prime = grad(g_prime)  # 12x^2

x = 2.0
print(f"g({x}) = {g(x)}")
print(f"g'({x}) = {g_prime(x)}")
print(f"g''({x}) = {g_double_prime(x)}")

## 6. Practice: Manual SGD with JAX

In [ ]:
key = random.PRNGKey(42)
key, k1, k2 = random.split(key, 3)

n, d = 200, 3
true_w = jnp.array([2.0, -1.0, 0.5])
X = random.normal(k1, shape=(n, d))
y = X @ true_w + 0.1 * random.normal(k2, shape=(n,))

params = {'w': jnp.zeros(d), 'b': jnp.array(0.0)}
lr = 0.1

@jit
def train_step(params, X, y):
  loss, grads = value_and_grad(loss_fn)(params, X, y)
  params = {
    'w': params['w'] - lr * grads['w'],
    'b': params['b'] - lr * grads['b'],
  }
  return params, loss

for step in range(100):
  params, loss = train_step(params, X, y)
  if step % 25 == 0:
    print(f"step {step:3d} | loss: {loss:.4f} | w: {params['w']}")

print(f"\ntrue w: {true_w}")
print(f"learned:  {params['w']}")